Un biolog studiază o populaţie de animale al cărei efectiv total N este necunoscut. Iniţial, sunt capturate
şi marcate M = 20 animale, care sunt apoi eliberate. La o a doua capturare sunt selectate aleatoriu n = 15
animale, dintre care Y = 4 sunt marcate. Presupunând că toate animalele au aceeaşi probabilitate de a fi
capturate, considerăm

Y | N ∼ Hypergeometric(N , M, n)
N ∼ DiscreteUniform(20, 200).

In [6]:
from math import comb
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm

# Date
M, n, Y = 20, 15, 4

a) Construiţi modelul în PyMC, determinaţi distribuţia a posteriori a lui N şi reprezentaţi-o grafic.

In [7]:
# Modelul bayesian
with pm.Model() as model:
  N = pm.DiscreteUniform("N", lower=20, upper=200)
  pm.HyperGeometric("Y_obs", N=N, k=M, n=n, observed=Y)
  idata = pm.sample(draws=4000, tune=2000, chains=4, random_seed=42)

# a) Distributia a posteriori a lui N
pc_post = az.plot_dist(
    idata,
    var_names=["N"],
    ci_kind="hdi",
    ci_prob=0.95,
    backend="matplotlib",
)
pc_post.savefig("1a_posterior_N.png", dpi=150, bbox_inches="tight")
plt.close("all")

Multiprocess sampling (4 chains in 4 jobs)
Metropolis: [N]


Output()

Sampling 4 chains for 2_000 tune and 4_000 draw iterations (8_000 + 16_000 draws total) took 5 seconds.


b) Estimaţi media, mediana şi intervalul HDI de 95% pentru N. Comparaţi media a posteriori cu estimarea
clasică Mn/Y.

In [8]:
# b) Media, mediana, HDI 95%
mean_N = idata.posterior["N"].mean().item()
median_N = idata.posterior["N"].median().item()
hdi_N = az.hdi(idata.posterior["N"], prob=0.95).values.ravel()

print(f"Media posterior N: {mean_N:.3f}")
print(f"Mediana posterior N: {median_N:.3f}")
print(f"HDI 95% N: [{hdi_N[0]:.0f}, {hdi_N[1]:.0f}]")
print(f"Estimare clasica Mn/Y: {M * n / Y:.3f}")

Media posterior N: 103.627
Mediana posterior N: 96.000
HDI 95% N: [43, 180]
Estimare clasica Mn/Y: 75.000


c) Estimaţi probabilitatea a posteriori P (N > 100 | Y = 4).

In [9]:
# c) P(N > 100 | Y = 4)
p_100 = (idata.posterior["N"] > 100).mean().item()
print(f"P(N > 100 | Y=4) = {p_100:.4f}")

P(N > 100 | Y=4) = 0.4596


d) Generaţi distribuţia predictivă a posteriori a numărului de animale marcate într-o nouă capturare de
15 animale, presupunând că marcajele rămân neschimbate. Explicaţi diferenţa faţă de distribuţia a
posteriori a lui N.

In [10]:
# d) Distributia predictiva a posteriori
with model:
  pm.HyperGeometric("y_new", N=N, k=M, n=n)
  ppc = pm.sample_posterior_predictive(idata, var_names=["y_new"], random_seed=42)

pc_ppc = az.plot_dist(
    ppc,
    var_names=["y_new"],
    group="posterior_predictive",
    ci_kind="hdi",
    ci_prob=0.95,
    backend="matplotlib",
)
pc_ppc.savefig("1d_posterior_predictive_y_new.png", dpi=150, bbox_inches="tight")
plt.close("all")

print(
    az.summary(
        ppc,
        var_names=["y_new"],
        group="posterior_predictive",
        ci_kind="hdi",
        ci_prob=0.95,
    )
)

# Verificare exacta
grid = np.arange(20, 201)
lik = np.array([
    comb(M, Y) * comb(N_val - M, n - Y) / comb(N_val, n) if N_val >= 31 else 0
    for N_val in grid
])
post_exact = lik / lik.sum()

print(f"Exact mean N: {(grid * post_exact).sum():.3f}")
print(f"Exact P(N > 100): {post_exact[grid > 100].sum():.4f}")

/home/eeemy/Work/Info/PMP-2026/.venv/lib/python3.14/site-packages/pytensor/link/numba/dispatch/basic.py:234: UserWarning: Numba will use object mode to run hypergeometric_rv{"(),(),()->()"}'s perform method. Set `pytensor.config.compiler_verbose = True` to see more details.
  warnings.warn(
Sampling: [y_new]


Output()

       mean   sd hdi95_lb hdi95_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
y_new  3.29  1.9        0        7     5942     8993  1.00     0.025   0.014
Exact mean N: 102.613
Exact P(N > 100): 0.4494


## Model

- Prior: `N ~ DiscreteUniform(20, 200)`
- Likelihood: `Y | N ~ HyperGeometric(N, M=20, n=15)`
- Date observate: `Y = 4`

## Fișiere generate

- `posterior_N.png` – distribuția a posteriori pentru `N`
- `posterior_predictive_y_new.png` – distribuția predictivă pentru numărul de animale marcate într-o nouă capturare de 15 animale

## Interpretare

Distribuția a posteriori pentru `N` arată incertitudinea privind efectivul total al populației după observarea a 4 animale marcate în a doua capturare. Deoarece în eșantion apar 11 animale nemarcate, valorile `N < 31` devin imposibile, deci posteriorul este concentrat pe valori mai mari.

Media și mediana posteriorului se compară cu estimarea:

$$E[X] = n \cdot \frac{K}{N}$$

`M * n / Y = 20 * 15 / 4 = 75`

Estimarea bayesiană este apropiată, dar oferă în plus un interval HDI de 95%, care cuantifică incertitudinea.

Probabilitatea `P(N > 100 | Y = 4)` se calculează direct din eșantioanele posteriorului și indică cât de plauzibil este ca populația să fie mai mare de 100.

Distribuția predictivă `y_new` reprezintă numărul de animale marcate într-o nouă capturare de 15 animale. Aceasta este diferită de distribuția lui `N`: `N` este dimensiunea populației, în timp ce `y_new` este un număr de animale marcate într-un eșantion, cu valori posibile între 0 și 15. Distribuția predictivă include atât incertitudinea asupra lui `N`, cât și variabilitatea aleatoare a noii capturări, deci este mai dispersată.